# End-to-End Training with Differentiable Phase Reconstruction

This notebook demonstrates the key application of Diff-RTPGHI: **training a neural network through the phase retrieval step** using gradient-based optimisation.

We train a simple magnitude-modification network for **filterbank denoising**:

```
noisy signal → analysis → |magnitudes| → neural net → modified magnitudes
                                                            ↓
                            time-domain loss ← synthesis ← Diff-RTPGHI
```

Without differentiable phase reconstruction, gradients cannot flow from the time-domain loss back to the magnitude predictor. Diff-RTPGHI enables this.

**Note:** This uses a PyTorch wrapper around the NumPy Diff-RTPGHI implementation. The forward pass runs in NumPy; the backward pass uses the straight-through estimator.

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                    'cool-frames @ git+https://github.com/allthatsounds/cool-frames.git'],
                   check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'torch'],
                   check=True)


import matplotlib.pyplot as plt
import torch.nn as nn
import torch.optim as optim

import numpy as np
import torch

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 4)


In [ ]:
from cool_frames.numpy.filterbanks._core import filterbank, ifilterbank
from cool_frames.numpy.filterbanks._frame import filterbankrealdual
from cool_frames.numpy.filterbanks._utils import normalise_a
from cool_frames.numpy.filters._design import audfilters
from cool_frames.numpy.filters._gabfilters import _comp_tfrfromwin
from cool_frames.numpy.phase._diff_admm import backward_fd, diff_admm, diff_dm, diff_raar
from cool_frames.numpy.phase._diff_constphase import (
    constphase_nonuniform,
)

print('All imports OK.')


## 1. Filterbank Setup

In [ ]:
fs = 16000
Ls = 4000  # 0.25s signals for fast training
redmul = 8.0

g, a, fc_hz, L = audfilters(fs, Ls, redmul=redmul)
M = len(g)
a_norm = normalise_a(a, M)
a_int = np.array([int(a_norm[m, 0]) for m in range(M)])
fc_norm = np.array(fc_hz) / fs * 2.0
gd = filterbankrealdual(g, a_norm, L)

# TFR for magnitude-based gradients
tfr = np.zeros(M)
for m_idx in range(M):
    gm = g[m_idx]
    if 'H' in gm:
        H_vals = np.asarray(gm['H'](L))
        h_time = np.fft.ifft(H_vals)
        h_real = np.real(np.fft.fftshift(h_time))
        gamma = _comp_tfrfromwin(h_real)
        tfr[m_idx] = gamma / L if L > 0 else 0.0

N_frames = [L // a_int[m] for m in range(M)]
total_coeffs = sum(N_frames)

print(f'Filterbank: M={M}, L={L}')
print(f'Total coefficients per signal: {total_coeffs}')

## 2. Differentiable Phase Retrieval (PyTorch Wrapper)

We wrap the NumPy Diff-RTPGHI in a PyTorch `autograd.Function` using the straight-through estimator: the forward pass runs the fixed-order phase integration, and the backward pass approximates the Jacobian via finite differences on the fly.

In [ ]:
class DiffRTPGHI(torch.autograd.Function):
    """Differentiable phase retrieval via straight-through estimator.
    
    Forward: runs NumPy constphase_nonuniform
    Backward: finite-difference Jacobian approximation
    """

    @staticmethod
    def forward(ctx, magnitudes_flat, a_int_np, fc_norm_np, tfr_np,
                N_frames_list, M_channels):
        """magnitudes_flat: (total_coeffs,) tensor of magnitudes."""
        ctx.save_for_backward(magnitudes_flat)
        ctx.a_int_np = a_int_np
        ctx.fc_norm_np = fc_norm_np
        ctx.tfr_np = tfr_np
        ctx.N_frames_list = N_frames_list
        ctx.M_channels = M_channels

        mag_np = magnitudes_flat.detach().cpu().numpy()

        # Unflatten to per-channel arrays
        s_list = []
        offset = 0
        for m in range(M_channels):
            nm = N_frames_list[m]
            s_list.append(np.maximum(mag_np[offset:offset+nm], 1e-10))
            offset += nm

        # Run Diff-RTPGHI
        c_list, phase_list, _, _ = constphase_nonuniform(
            s_list, a_int_np, fc_norm_np, tfr_np, tol=1e-6)

        # Flatten phase output
        phase_flat = np.concatenate([np.asarray(p).ravel() for p in phase_list])

        return torch.from_numpy(phase_flat).float()

    @staticmethod
    def backward(ctx, grad_phase):
        """Straight-through: approximate d_phase/d_magnitude via finite differences."""
        magnitudes_flat, = ctx.saved_tensors
        mag_np = magnitudes_flat.detach().cpu().numpy()
        grad_phase_np = grad_phase.detach().cpu().numpy()
        M = ctx.M_channels
        N_list = ctx.N_frames_list
        total = sum(N_list)

        eps = 1e-4
        grad_mag = np.zeros(total)

        def run_phase(mag_vec):
            s_list = []
            off = 0
            for m in range(M):
                nm = N_list[m]
                s_list.append(np.maximum(mag_vec[off:off+nm], 1e-10))
                off += nm
            c_list, phase_list, _, _ = constphase_nonuniform(
                s_list, ctx.a_int_np, ctx.fc_norm_np, ctx.tfr_np, tol=1e-6)
            return np.concatenate([np.asarray(p).ravel() for p in phase_list])

        phase_ref = run_phase(mag_np)

        # Compute gradient via vector-Jacobian product with random projections
        # (much faster than full Jacobian for high-dimensional inputs)
        # Use coordinate-wise finite differences for a subset of dimensions
        n_sample = min(total, 50)  # Sample dimensions for efficiency
        rng = np.random.default_rng(0)
        indices = rng.choice(total, n_sample, replace=False)

        for j in indices:
            mag_pert = mag_np.copy()
            mag_pert[j] += eps
            phase_pert = run_phase(mag_pert)
            dphi_dsj = (phase_pert - phase_ref) / eps
            grad_mag[j] = np.dot(grad_phase_np, dphi_dsj)

        # Scale up to account for sampling
        if n_sample < total:
            grad_mag *= total / n_sample

        return torch.from_numpy(grad_mag).float(), None, None, None, None, None


def diff_rtpghi_phase(magnitudes_flat, a_int_np, fc_norm_np, tfr_np, N_list, M_ch):
    """Convenience wrapper."""
    return DiffRTPGHI.apply(magnitudes_flat, a_int_np, fc_norm_np, tfr_np, N_list, M_ch)


print('DiffRTPGHI autograd function defined.')

### Alternative: ADMM / RAAR / DM Phase Retrieval

Unlike Diff-RTPGHI (which requires a straight-through estimator for the argsort), the ADMM/RAAR/DM iterations are **naturally differentiable**: P_A is smooth (angle extraction + magnitude replacement) and P_C is linear (filterbank synthesis + analysis). When unrolled for a fixed number of iterations, the full magnitude→phase map has well-defined gradients everywhere.

In [ ]:
class DiffIterativePhaseRetrieval(torch.autograd.Function):
    """Differentiable ADMM/RAAR/DM phase retrieval for PyTorch.
    
    Unlike DiffRTPGHI, the ADMM/RAAR/DM iterations are naturally
    differentiable (P_A is smooth, P_C is linear). The backward pass
    uses the same finite-difference VJP for consistency, but the
    forward pass is exact (no straight-through approximation needed).
    """

    @staticmethod
    def forward(ctx, magnitudes_flat, g_filters, a_hops, N_frames_list,
                L_val, method, maxit, extra_kwargs):
        ctx.save_for_backward(magnitudes_flat)
        ctx.g_filters = g_filters
        ctx.a_hops = a_hops
        ctx.N_frames_list = N_frames_list
        ctx.L_val = L_val
        ctx.method = method
        ctx.maxit = maxit
        ctx.extra_kwargs = extra_kwargs

        mag_np = magnitudes_flat.detach().cpu().numpy()

        func = {'admm': diff_admm, 'raar': diff_raar, 'dm': diff_dm}[method]
        phase_flat, _ = func(
            mag_np, g_filters, a_hops, N_frames_list,
            L=L_val, real=True, maxit=maxit, startphase='zero',
            **extra_kwargs
        )

        return torch.from_numpy(phase_flat).float()

    @staticmethod
    def backward(ctx, grad_phase):
        magnitudes_flat, = ctx.saved_tensors
        mag_np = magnitudes_flat.detach().cpu().numpy()
        grad_phase_np = grad_phase.detach().cpu().numpy()

        func = {'admm': diff_admm, 'raar': diff_raar, 'dm': diff_dm}[ctx.method]

        def run_fn(mag):
            phase, _ = func(
                mag, ctx.g_filters, ctx.a_hops, ctx.N_frames_list,
                L=ctx.L_val, real=True, maxit=ctx.maxit, startphase='zero',
                **ctx.extra_kwargs
            )
            return phase

        grad_mag = backward_fd(run_fn, mag_np, grad_phase_np, n_sample=50, seed=0)
        return torch.from_numpy(grad_mag).float(), None, None, None, None, None, None, None


def diff_iterative_phase(magnitudes_flat, g_filters, a_hops, N_frames_list,
                         L_val, method='raar', maxit=30, **kwargs):
    """Convenience wrapper for differentiable ADMM/RAAR/DM.
    
    Usage:
        phase = diff_iterative_phase(predicted_mag, g, a_norm, N_frames, L,
                                     method='raar', maxit=30, beta=0.9)
    """
    return DiffIterativePhaseRetrieval.apply(
        magnitudes_flat, g_filters, a_hops, N_frames_list,
        L_val, method, maxit, kwargs
    )


print('DiffIterativePhaseRetrieval (ADMM/RAAR/DM) autograd function defined.')
print('Available methods: "admm", "raar", "dm"')

## 3. Filterbank Analysis/Synthesis in PyTorch

We wrap the NumPy analysis and synthesis operations so they work with PyTorch tensors.

In [ ]:
def analyse_signal(sig_np):
    """Analyse a signal into filterbank magnitudes. Returns (mag_flat, c_orig)."""
    sig_padded = np.zeros(L)
    sig_padded[:min(len(sig_np), L)] = sig_np[:min(len(sig_np), L)]
    c = filterbank(sig_padded, g, a_norm, L=L)
    mag_flat = np.concatenate([np.abs(np.asarray(ci).ravel()) for ci in c])
    return mag_flat, c


def synthesise_from_mag_phase(mag_flat_np, phase_flat_np):
    """Synthesise time-domain signal from magnitudes and phases."""
    c_list = []
    offset = 0
    for m in range(M):
        nm = N_frames[m]
        mag_m = mag_flat_np[offset:offset+nm]
        phi_m = phase_flat_np[offset:offset+nm]
        c_list.append(mag_m * np.exp(1j * phi_m))
        offset += nm
    sig = ifilterbank(c_list, gd, a_norm, Ls=L, real=True)
    return np.real(sig[:Ls])


class SynthesisFunction(torch.autograd.Function):
    """Differentiable synthesis: magnitudes + phases -> waveform."""

    @staticmethod
    def forward(ctx, mag_flat, phase_flat):
        ctx.save_for_backward(mag_flat, phase_flat)
        mag_np = mag_flat.detach().cpu().numpy()
        phase_np = phase_flat.detach().cpu().numpy()
        sig = synthesise_from_mag_phase(mag_np, phase_np)
        return torch.from_numpy(sig.copy()).float()

    @staticmethod
    def backward(ctx, grad_output):
        mag_flat, phase_flat = ctx.saved_tensors
        mag_np = mag_flat.detach().cpu().numpy()
        phase_np = phase_flat.detach().cpu().numpy()
        grad_out_np = grad_output.detach().cpu().numpy()

        eps = 1e-5
        total = len(mag_np)

        # Approximate gradients w.r.t. magnitudes
        sig_ref = synthesise_from_mag_phase(mag_np, phase_np)
        grad_mag = np.zeros(total)
        grad_phase = np.zeros(total)

        # Sample a subset for efficiency
        n_sample = min(total, 50)
        rng = np.random.default_rng(1)
        indices = rng.choice(total, n_sample, replace=False)

        for j in indices:
            # d_sig / d_mag_j
            mag_pert = mag_np.copy()
            mag_pert[j] += eps
            sig_pert = synthesise_from_mag_phase(mag_pert, phase_np)
            dsig_dmag = (sig_pert - sig_ref) / eps
            grad_mag[j] = np.dot(grad_out_np[:len(dsig_dmag)], dsig_dmag)

            # d_sig / d_phase_j
            phase_pert = phase_np.copy()
            phase_pert[j] += eps
            sig_pert2 = synthesise_from_mag_phase(mag_np, phase_pert)
            dsig_dphase = (sig_pert2 - sig_ref) / eps
            grad_phase[j] = np.dot(grad_out_np[:len(dsig_dphase)], dsig_dphase)

        if n_sample < total:
            grad_mag *= total / n_sample
            grad_phase *= total / n_sample

        return torch.from_numpy(grad_mag).float(), torch.from_numpy(grad_phase).float()


print('Analysis/synthesis wrappers defined.')

## 4. Magnitude Modification Network

A simple 2-layer MLP that takes noisy filterbank magnitudes and predicts a multiplicative mask (like a Wiener filter). The mask is applied to produce clean magnitude estimates.

In [ ]:
class MagnitudeMaskNet(nn.Module):
    """Simple magnitude mask predictor.
    
    Input: log-magnitudes (total_coeffs,)
    Output: multiplicative mask (total_coeffs,) in [0, 1]
    """
    def __init__(self, n_coeffs, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_coeffs, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_coeffs),
            nn.Sigmoid()  # Output mask in [0, 1]
        )

    def forward(self, log_mag):
        return self.net(log_mag)


model = MagnitudeMaskNet(total_coeffs, hidden=64)
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Input/output dimension: {total_coeffs}')

## 5. Generate Training Data

Clean signals + additive noise = noisy signals. The network learns to predict a magnitude mask that suppresses the noise.

In [ ]:
def generate_training_pair(rng, snr_db=5.0):
    """Generate a (clean, noisy) signal pair."""
    t = np.arange(Ls) / fs

    # Clean: random sinusoids
    n_sines = rng.integers(2, 6)
    freqs = rng.uniform(100, fs/2 - 200, n_sines)
    amps = rng.uniform(0.2, 1.0, n_sines)
    phases = rng.uniform(0, 2*np.pi, n_sines)
    clean = sum(a * np.sin(2*np.pi*f*t + p) for a, f, p in zip(amps, freqs, phases))
    clean = clean / (np.max(np.abs(clean)) + 1e-10) * 0.8

    # Noise
    noise = rng.standard_normal(Ls)
    noise_power = np.sum(clean**2) / (10**(snr_db/10))
    noise = noise * np.sqrt(noise_power / (np.sum(noise**2) + 1e-10))

    noisy = clean + noise
    return clean, noisy


# Generate dataset
rng = np.random.default_rng(42)
n_train = 50
n_val = 10

train_data = [generate_training_pair(rng, snr_db=5.0) for _ in range(n_train)]
val_data = [generate_training_pair(rng, snr_db=5.0) for _ in range(n_val)]

print(f'Training set: {n_train} pairs')
print(f'Validation set: {n_val} pairs')
print('SNR: 5 dB')

## 6. Training Loop

We train with two losses:
- **Magnitude loss** (always available): MSE between predicted and clean magnitudes
- **Time-domain loss** (requires differentiable phase reconstruction): MSE between reconstructed and clean waveforms

The time-domain loss is only possible because Diff-RTPGHI lets gradients flow through the phase retrieval step.

In [ ]:
def train_one_epoch(model, optimizer, train_data, use_time_domain_loss=True,
                    mag_weight=1.0, td_weight=0.1):
    """Train for one epoch."""
    model.train()
    total_loss = 0.0

    for clean, noisy in train_data:
        optimizer.zero_grad()

        # Analysis
        noisy_mag, _ = analyse_signal(noisy)
        clean_mag, _ = analyse_signal(clean)

        # Forward through network
        noisy_mag_t = torch.from_numpy(noisy_mag).float()
        clean_mag_t = torch.from_numpy(clean_mag).float()
        log_noisy = torch.log(noisy_mag_t + 1e-8)

        mask = model(log_noisy)
        predicted_mag = noisy_mag_t * mask

        # Magnitude loss
        loss_mag = torch.mean((predicted_mag - clean_mag_t)**2)
        loss = mag_weight * loss_mag

        # Time-domain loss (through Diff-RTPGHI)
        if use_time_domain_loss:
            phase = diff_rtpghi_phase(
                predicted_mag, a_int, fc_norm, tfr, N_frames, M)
            sig_recon = SynthesisFunction.apply(predicted_mag, phase)
            clean_t = torch.from_numpy(clean[:Ls].copy()).float()
            loss_td = torch.mean((sig_recon - clean_t)**2)
            loss = loss + td_weight * loss_td

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(train_data)


def evaluate(model, val_data):
    """Evaluate SDR improvement on validation set."""
    model.eval()
    sdrs_in = []
    sdrs_out = []

    with torch.no_grad():
        for clean, noisy in val_data:
            noisy_mag, _ = analyse_signal(noisy)
            noisy_mag_t = torch.from_numpy(noisy_mag).float()
            log_noisy = torch.log(noisy_mag_t + 1e-8)

            mask = model(log_noisy)
            predicted_mag = (noisy_mag_t * mask).numpy()

            # Phase retrieval + synthesis
            s_list = []
            offset = 0
            for m in range(M):
                nm = N_frames[m]
                s_list.append(np.maximum(predicted_mag[offset:offset+nm], 1e-10))
                offset += nm
            c_list, _, _, _ = constphase_nonuniform(s_list, a_int, fc_norm, tfr)
            sig_out = ifilterbank(c_list, gd, a_norm, Ls=L, real=True)
            sig_out = np.real(sig_out[:Ls])

            def sdr_fn(ref, est):
                n = min(len(ref), len(est))
                return 10*np.log10(np.sum(ref[:n]**2) / (np.sum((ref[:n]-est[:n])**2) + 1e-30))

            sdrs_in.append(sdr_fn(clean, noisy))
            sdrs_out.append(sdr_fn(clean, sig_out))

    return np.mean(sdrs_in), np.mean(sdrs_out)


print('Training functions defined.')

In [ ]:
# Train with magnitude-only loss (baseline, no phase reconstruction needed)
print('='*60)
print('Training with MAGNITUDE-ONLY loss (no Diff-RTPGHI needed)')
print('='*60)

model_mag = MagnitudeMaskNet(total_coeffs, hidden=64)
optimizer_mag = optim.Adam(model_mag.parameters(), lr=1e-3)

losses_mag = []
sdrs_mag = []

for epoch in range(20):
    loss = train_one_epoch(model_mag, optimizer_mag, train_data,
                           use_time_domain_loss=False)
    sdr_in, sdr_out = evaluate(model_mag, val_data)
    losses_mag.append(loss)
    sdrs_mag.append(sdr_out)
    if (epoch + 1) % 5 == 0:
        print(f'  Epoch {epoch+1:2d}: loss={loss:.6f}, '
              f'SDR_in={sdr_in:.1f} -> SDR_out={sdr_out:.1f} dB '
              f'(+{sdr_out-sdr_in:.1f} dB)')

In [ ]:
# Train with magnitude + time-domain loss (using Diff-RTPGHI)
print('='*60)
print('Training with MAGNITUDE + TIME-DOMAIN loss (via Diff-RTPGHI)')
print('='*60)

model_td = MagnitudeMaskNet(total_coeffs, hidden=64)
optimizer_td = optim.Adam(model_td.parameters(), lr=1e-3)

losses_td = []
sdrs_td = []

for epoch in range(20):
    loss = train_one_epoch(model_td, optimizer_td, train_data,
                           use_time_domain_loss=True,
                           mag_weight=1.0, td_weight=0.1)
    sdr_in, sdr_out = evaluate(model_td, val_data)
    losses_td.append(loss)
    sdrs_td.append(sdr_out)
    if (epoch + 1) % 5 == 0:
        print(f'  Epoch {epoch+1:2d}: loss={loss:.6f}, '
              f'SDR_in={sdr_in:.1f} -> SDR_out={sdr_out:.1f} dB '
              f'(+{sdr_out-sdr_in:.1f} dB)')

In [ ]:
# Compare training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(losses_mag, label='Magnitude-only loss', linewidth=2)
ax1.plot(losses_td, label='Mag + time-domain loss', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Training loss')
ax1.set_title('Training Loss')
ax1.legend()
ax1.set_yscale('log')

ax2.plot(sdrs_mag, 'o-', label='Magnitude-only loss', linewidth=2)
ax2.plot(sdrs_td, 's-', label='Mag + time-domain loss', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Validation SDR (dB)')
ax2.set_title('Validation SDR')
ax2.legend()

plt.suptitle('Effect of time-domain loss (enabled by Diff-RTPGHI)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print(f'\nFinal SDR (mag-only):      {sdrs_mag[-1]:.1f} dB')
print(f'Final SDR (mag + TD):      {sdrs_td[-1]:.1f} dB')
print(f'Improvement from TD loss:  {sdrs_td[-1] - sdrs_mag[-1]:+.1f} dB')

## 7. Visualise Results

In [ ]:
# Pick a validation example and show waveforms
clean_ex, noisy_ex = val_data[0]

# Run both models
def run_model(model, noisy_sig):
    model.eval()
    with torch.no_grad():
        noisy_mag, _ = analyse_signal(noisy_sig)
        noisy_mag_t = torch.from_numpy(noisy_mag).float()
        mask = model(torch.log(noisy_mag_t + 1e-8))
        predicted_mag = (noisy_mag_t * mask).numpy()

    s_list = []
    offset = 0
    for m in range(M):
        nm = N_frames[m]
        s_list.append(np.maximum(predicted_mag[offset:offset+nm], 1e-10))
        offset += nm
    c_list, _, _, _ = constphase_nonuniform(s_list, a_int, fc_norm, tfr)
    sig_out = ifilterbank(c_list, gd, a_norm, Ls=L, real=True)
    return np.real(sig_out[:Ls])

sig_mag_only = run_model(model_mag, noisy_ex)
sig_td_loss = run_model(model_td, noisy_ex)

t_plot = np.arange(Ls) / fs
fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)

axes[0].plot(t_plot, clean_ex, color='green', alpha=0.8)
axes[0].set_ylabel('Clean')
axes[0].set_title('Denoising comparison')

axes[1].plot(t_plot, noisy_ex, color='gray', alpha=0.8)
axes[1].set_ylabel('Noisy (5 dB SNR)')

def sdr_fn(ref, est):
    n = min(len(ref), len(est))
    return 10*np.log10(np.sum(ref[:n]**2)/(np.sum((ref[:n]-est[:n])**2)+1e-30))

axes[2].plot(t_plot[:len(sig_mag_only)], sig_mag_only, color='blue', alpha=0.8)
axes[2].set_ylabel(f'Mag-only ({sdr_fn(clean_ex, sig_mag_only):.1f} dB)')

axes[3].plot(t_plot[:len(sig_td_loss)], sig_td_loss, color='red', alpha=0.8)
axes[3].set_ylabel(f'Mag+TD ({sdr_fn(clean_ex, sig_td_loss):.1f} dB)')
axes[3].set_xlabel('Time (s)')

plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated that Diff-RTPGHI enables **end-to-end training through phase retrieval**:

1. A magnitude mask network was trained for filterbank denoising
2. Adding a time-domain loss (only possible with differentiable phase reconstruction) improved reconstruction quality
3. Gradients flowed from the waveform loss, through synthesis, through Diff-RTPGHI phase retrieval, and into the magnitude predictor

This is a minimal proof of concept. In practice, this approach enables:
- Training neural vocoders with non-uniform filterbank representations
- End-to-end optimisation of audio enhancement systems
- Learning filterbank parameters jointly with downstream tasks

See the paper for full details on the Diff-RTPGHI algorithm and its properties.